# Гидравлические расчеты
Выполняет предварительные гидрологические расчеты

In [5]:
import math
from enum import Flag, auto
from pathlib import Path

import arcpy
from arcpy.sa import Con, Raster, IsNull, RemapValue, Reclassify, Square, Power, SquareRoot, VfTable

class РасчетнаяЗадача(Flag):
    пустышка = auto()
    гидро_коррекция_цмр = auto()
    уклон = auto()
    сток_направление = auto()
    scs_cn = auto()
    сток_rx5day = auto()
    сток_накопление_rx5day = auto()
    ливень_интенсивность = auto()
    ливень_расход = auto()
    ливень_глубина_стока = auto()
    водотоки_выявить_rx5day = auto()
    водотоки_rx5day = auto()
    водотоки_векторизовать_rx5day = auto()
    водотоки_и_океан = auto()
    шероховатость = auto()
    поверхность_стоимости = auto()
    вертикальный_фактор = auto()
    накопление_расстояния = auto()
    цмр_прожиг = auto()    
    hand = auto()
    бассейны = auto()
    бассейны_веторизовать = auto()

# === Исходные пути ===
arcpy.env.overwriteOutput = True
temp = r"F:\temp"
затопление_бд = r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\FloodFactors.gdb"
покрытие_бд = r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\WorldCover.gdb"
цмр_дб = r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\TanDEMX.gdb"
растры_папка = r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\Rasters"

расчет_hand_инструмент_путь = r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\Scripts\Topography_Toolbox_Pro\Topography Toolbox Pro31.atbx"

hydro_soil = rf"{растры_папка}\HiHydroSoil.tif"     # Растр гидрологических групп почв HiHydroSoil
tandem = rf"{цмр_дб}\TDX_EGM_Mosaic"                # Растр ЦМР TanDEM-X
океан = rf"{цмр_дб}\ocean_border"                   # Границы океана
покрытие = rf"{покрытие_бд}\WorldCover_Mozaic"      # Растр землепользований и покрытий WorldCover
осадки_rx5day = rf"{затопление_бд}\Rx5DayAri40"     # Растр максимальных осадков за 5 суток (мм)
осадки_rx1day = rf"{затопление_бд}\Rx1DayAri40"     # Растр максимальных суточных осадков (мм)
бассейны_генер = rf"{затопление_бд}\bassin_v_gen_clip"# Векторный слой генерализованных бассейнов
стоимость_гор_тайлы = rf"{temp}\acc\cost_tiles"     # Папка для растров стоимости разрезанных по границам бассейнов
стоимость_накопл_тайлы = rf"{temp}\acc\acc_tiles"   # Папка для растров накопленной стоимости по границам бассейнов
таблица_VF_название = "vf_table.txt"

# === Расчетные параметры ===

# Что будем рассчитывать
задача: РасчетнаяЗадача = (РасчетнаяЗадача.пустышка
                           # | РасчетнаяЗадача.гидро_коррекция_цмр
                           # | РасчетнаяЗадача.уклон
                           # | РасчетнаяЗадача.сток_направление
                           # | РасчетнаяЗадача.scs_cn
                           # | РасчетнаяЗадача.сток_rx5day
                           # | РасчетнаяЗадача.сток_накопление_rx5day
                           # | РасчетнаяЗадача.ливень_интенсивность
                           # | РасчетнаяЗадача.ливень_расход
                           # | РасчетнаяЗадача.ливень_глубина_стока
                           # | РасчетнаяЗадача.водотоки_выявить_rx5day
                           # | РасчетнаяЗадача.водотоки_rx5day
                           # | РасчетнаяЗадача.водотоки_векторизовать_rx5day
                           # | РасчетнаяЗадача.водотоки_и_океан
                           # | РасчетнаяЗадача.бассейны
                           # | РасчетнаяЗадача.бассейны_веторизовать
                           # | РасчетнаяЗадача.шероховатость
                           # | РасчетнаяЗадача.поверхность_стоимости
                           # | РасчетнаяЗадача.вертикальный_фактор
                           # | РасчетнаяЗадача.накопление_расстояния
                           # | РасчетнаяЗадача.цмр_прожиг # Не пригодился
                           # | РасчетнаяЗадача.hand
                           )

# --- Параметры дождя ---
дождь_расчетный_продолжительность = 1.          # Расчетное продолжительность дождя (часы)
осадки_за_расчетное_время_доля = 0.32           # Доля осадков, выпадающая за время дождя
пик_дождя_интенсив = 3                          # Учитывает, что в пределах часа дождя есть 15-20 минут, формирующих максимальный сток

# --- Параметры стока ---
сток_руслоформирования = 7561                   # Поток, после которого захватываем водоток
ia_k = 0.05                                     # Коэффициент начальных потерь (обычно 0.2) для расчета по методике SCS-CN

# --- Параметры поверхности стоимости затопления ---
шероховатость_база = 0.033
поток_глубина_цель = 0.2                                                # Глубина начала ущерба (м)
поток_скорость_цель = math.sqrt(9.81 * поток_глубина_цель) * .5         # Целевая скорость потока (м/с) для расчета поверхности стоимости (½ числа Фруда)
гидро_уклон_база = (math.pow(шероховатость_база, 2) * math.pow(поток_скорость_цель, 2)) / math.pow(поток_глубина_цель, 4. / 3.)


# === Выходные данные ===
цмр = rf"{цмр_дб}\tdx_egm_fill"                             # Растр откорректированной ЦМР, используемой в расчетах
уклон = rf"{цмр_дб}\slope_percent"                          # Растр уклона в %
scs_cn = rf"{покрытие_бд}\SCS_CN"                           # Растр кривых стока (SCS-CN)
сток_rx5day_мм = rf"{затопление_бд}\runoff_rx5day_mm"       # Растр стока с ячеки при rx5day, мм
сток_rx5day_м = rf"{затопление_бд}\runoff_rx5day_m"         # Растр стока с ячеки при rx5day, м
сток_направление = rf"{затопление_бд}\flow_dir"             # Растр направления стока
сток_накопление_rx5day = rf"{затопление_бд}\flow_acc_rx5day"# Растр накопления стока
водотоки_неупоряд_rx5day = rf"{затопление_бд}\watercourse"  # Растр водотоков
водотоки_rx5day = rf"{затопление_бд}\stream_rx5day"         # Растр упорядоченных водотоков
водотоки_rx5day_v = rf"{затопление_бд}\stream_v"            # Векторный слой основных водотоков
водотоки_и_океан = rf"{затопление_бд}\stream_with_ocean"    # Векторный слой основных водотоков

bassin = rf"{затопление_бд}\bassin"                         # Растр бассейнов
bassin_v = rf"{затопление_бд}\bassin_v"                     # Векторный слой бассейнов

маннинг = rf"{покрытие_бд}\manning_n"                       # Растр коэффициентов шероховатости n
цмр_burn = rf"{цмр_дб}\tdx_egm_fill_burn"                   # Растр ЦМР с прожженными водотоками

hand = rf"{затопление_бд}\HAND"                             # Растр высот над водотоками

ливень_интенсивность = rf"{затопление_бд}\rainfall_intensity_mms"       # Интенсивность ливня (мм/с)
ливень_расход_удельный = rf"{затопление_бд}\discharge"                  # Удельный расход (мм × м/с). Миллиметры не перевел в метро, чтобы не потерять точность
глубина = rf"{затопление_бд}\depth_h"                                   # Глубина стока во время ливня (м)
стоимость_горизонтальная = rf"{затопление_бд}\cost_horizont"            # Поверхность стоимости (гидравлический уклон)
накопленный_напор = rf"{затопление_бд}\pressure_acc"                    # Итоговый растр накопления
таблица_VF = str(Path(temp) / таблица_VF_название)                      # Таблица вертикального фактора



# === Справочники ===
# --- Таблица коэффициентов шероховатасти поверхности для WorldCover ---
# Источник: Manning's Roughness Generator (Medium n)
# Формат: [Код WorldCover, Маннинг n]
покрытие_шероховатость_перекодировка = [
    [10, 0.094],  # Деревья
    [20, 0.066],  # Кустарники
    [30, 0.033],  # Луг
    [40, 0.035],  # Пашня
    [50, 0.060],  # Застройка (ВАЖНО: Исправлено с 0.018 на 0.06 для реалистичности)
    [60, 0.033],  # Грунт
    [70, 0.020],  # Снег
    [80, 0.035],  # Вода
    [90, 0.090],  # Затапливаемые
    [95, 0.225],  # Мангры
    [100, 0.060]  # Мхи, лишайники
]

# [Класс покрытия × 10 + Группа почвы, значение CN]
покрытие_hydro_soil_csc_перекодировка = [
    [101, 36], [102, 60], [103, 73],  [104, 79],    # Деревья
    [201, 55], [202, 72], [203, 81],  [204, 86],    # Кустарники
    [301, 49], [302, 69], [303, 79],  [304, 84],    # Луг
    [401, 70], [402, 80], [403, 87],  [404, 90],    # Пашня по r.curvenumber
    [501, 89], [502, 92], [503, 94],  [504, 95],    # Застройка
    [601, 65], [602, 79], [603, 87],  [604, 90],    # Грунт
    [701, 100],[702, 100],[703, 100], [704, 100],   # Снег. Нули превращены в 100, чтобы избежать деления на 0
    [801, 100],[802, 100],[803, 100], [804, 100],   # Вода
    [901, 80], [902, 80], [903, 80],  [904, 80],    # Затапливаемые
    [951, 100],[952, 100],[953, 100], [954, 100],   # Мангры. Нули превращены в 100, чтобы избежать деления на 0
    [1001, 74], [1002, 77],[1003, 78],[1004, 79]    # Мхи
]

In [6]:
def create_vf_table(i_base, path : Path) -> str:
    """
    Создает ASCII таблицу для Вертикального Фактора по формуле:
    VF = 
        \begin{cases} 
            \frac {\sin(\alpha)}{i} + 1, & \text{если } VRMA \geq 0^\circ \\
            \sin(\alpha) + 1, & \text{если } VRMA < 0^\circ \\
            0.00001, & \text{если } VRMA \to -90^\circ \\
        \end{cases}

    """
    
    print("Таблица вертикального фактора:")
    
    with path.open("w") as f:
        # Структура файла Angle Factor
        # Генерируем значения от -90 до 90 градусов
        for deg in range(-90, 91):
            rad = math.radians(deg)

            if deg < 0:
                vf = math.sin(rad) + 1
            else:
                vf = (math.sin(rad) / i_base) + 1
            
            # Защита от обнуления, чтобы где-то внутри недр ArcGis не получить деление на 0
            if vf < 0.00001:
                vf = 0.00001

            f.write(f"{deg} {vf:.5f}\n")
            print(f"{deg} {vf:.5f}\n")

    print(f"Таблица вертикального фактора записана в файл: {path}")
    return str(path)

def calc_runoff(precipitation_raster: Raster, scs_cn_raster: Raster, ia_k: float):
    """
    Рассчитывает сток с ячейки
    :param precipitation_raster: растр осадков 
    :param scs_cn_raster: кривые стока SCS
    :param ia_k: коэффициент нальной абсорбции
    :return: сток с ячейки, мм
    """
    # S = Potential Maximum Retention
    retention_s = (25400. / scs_cn_raster) - 254.

    # Ia = Initial Abstraction
    initial_abstraction = ia_k * retention_s

    numerator = Square(precipitation_raster - initial_abstraction)
    denominator = precipitation_raster + (1 - ia_k) * retention_s

    # Основной расчет Runoff (мм)
    # Логика:
    # Если CN == 100, то сток = осадкам.
    # Иначе: Если Осадки > Начальных потерь, считаем по формуле, иначе 0.

    runoff_mm = Con(
        scs_cn_raster == 100,
        precipitation_raster,
        Con(
            precipitation_raster > initial_abstraction,
            numerator / denominator,
            0
        )
    )

    return runoff_mm

def get_arcpy_pixel_type(raster_path: str | Path) -> str:
    r = Raster(str(raster_path))
    pt = r.pixelType

    # Словарь сопоставления коротких кодов и полных имен для MosaicToNewRaster
    mapping = {
        'U1': '1_BIT',
        'U2': '2_BIT',
        'U4': '4_BIT',
        'U8': '8_BIT_UNSIGNED',
        'S8': '8_BIT_SIGNED',
        'U16': '16_BIT_UNSIGNED',
        'S16': '16_BIT_SIGNED',
        'U32': '32_BIT_UNSIGNED',
        'S32': '32_BIT_SIGNED',
        'F32': '32_BIT_FLOAT',
        'F64': '64_BIT', # В документации иногда 64_BIT_DOUBLE, но часто просто 64_BIT
    }

    # Для Distance Accumulation результат всегда Float, поэтому F32 или F64
    return mapping.get(pt, "32_BIT_FLOAT") # По умолчанию Float

In [7]:
with ((arcpy.EnvManager(outputCoordinateSystem=arcpy.SpatialReference(32631), cellSize=цмр, scratchWorkspace=temp))):
    num = 0

    # Гидрокоррекция растра — заполнение локальных понижений
    if РасчетнаяЗадача.гидро_коррекция_цмр in задача:
        out_surface_raster = arcpy.sa.Fill(
            in_surface_raster=tandem,
            z_limit=None
        )
        out_surface_raster.save(цмр)
        arcpy.management.BuildPyramids(цмр, -1, "NONE", "NEAREST", "DEFAULT", 75, "OVERWRITE")
        out_surface_raster = None
        num += 1
        print(f"{num}. Осуществлена гидрологическая коррекция растра")

    # Уклон поверхности в процентах
    if РасчетнаяЗадача.уклон in задача:
        arcpy.ddd.SurfaceParameters(
            in_raster=цмр,
            out_raster=уклон,
            parameter_type="SLOPE",
            local_surface_type="QUADRATIC",
            neighborhood_distance=None,
            use_adaptive_neighborhood="ADAPTIVE_NEIGHBORHOOD",
            z_unit="METER",
            output_slope_measurement="PERCENT_RISE",
            project_geodesic_azimuths="GEODESIC_AZIMUTHS",
            use_equatorial_aspect="NORTH_POLE_ASPECT",
            in_analysis_mask=None
        )
        arcpy.management.BuildPyramids(уклон, -1, "NONE", "NEAREST", "DEFAULT", 75, "OVERWRITE")
        num += 1
        print(f"{num}. Рассчитан уклон поверхности")

    # Расчет кривых стока
    if РасчетнаяЗадача.scs_cn in задача:
        land = Raster(покрытие)
        soil = Raster(hydro_soil)
        
        # Препроцессинг почвы (замена двойных групп)
        soil_clean = Con(soil == 14, 3, Con(soil == 24, 3, Con(soil == 34, 4, soil)))
        
        # Создаем индекс
        index_raster = (land * 10) + soil_clean
        
        cn_raster = Reclassify(index_raster, "Value", RemapValue(покрытие_hydro_soil_csc_перекодировка))
        cn_raster.save(scs_cn)
        arcpy.management.BuildPyramids(scs_cn, -1, "NONE", "NEAREST", "DEFAULT", 75, "OVERWRITE")
        cn_raster = None
        index_raster = None
        num += 1
        print(f"{num}. Рассчитаны кривые стока SCS-CN")

    # Расчет стока rx5day
    if РасчетнаяЗадача.сток_rx5day in задача:
        cn_raster = Raster(scs_cn)
        precip_rx5day_raster = Raster(осадки_rx5day)   
    
        runoff_raster_rx5day_mm = calc_runoff(precip_rx5day_raster, cn_raster, ia_k)
        # Пересчет в метры
        runoff_raster_rx5day_m = runoff_raster_rx5day_mm / 1000.0
 
        runoff_raster_rx5day_mm.save(сток_rx5day_мм)
        runoff_raster_rx5day_m.save(сток_rx5day_м)
        arcpy.management.BuildPyramids(сток_rx5day_мм, -1, "NONE", "NEAREST", "DEFAULT", 75, "OVERWRITE")
        arcpy.management.BuildPyramids(сток_rx5day_м, -1, "NONE", "NEAREST", "DEFAULT", 75, "OVERWRITE")
        cn_raster = None;precip_rx5day_raster = None;
        runoff_raster_rx5day_mm = None;runoff_raster_rx5day_m = None
        num += 1
        print(f"{num}. Рассчитан сток rx5day с ячейки")

    # Расчет интенсивности ливня
    if РасчетнаяЗадача.ливень_интенсивность in задача:
        дождь_расчетный_продолжительность_сек = дождь_расчетный_продолжительность * 3600
        
        cn_raster = Raster(scs_cn)
        precip_rx1day_raster = Raster(осадки_rx1day)

        # Расчетный объем осадков за время `дождь_расчетный_продолжительность`
        rainfall = precip_rx1day_raster * осадки_за_расчетное_время_доля

        # Сток с ячейки
        runoff_raster_mm = calc_runoff(rainfall, cn_raster, ia_k)
        mean_intensity_raster = runoff_raster_mm / дождь_расчетный_продолжительность_сек

        # Учитывает, что в пределах ливня есть самый мощный момент (пик), который кратно интенсивнее следующего период
        runoff_intensity_raster = mean_intensity_raster * пик_дождя_интенсив
        
        runoff_intensity_raster.save(ливень_интенсивность)
        arcpy.management.BuildPyramids(ливень_интенсивность, -1, "NONE", "NEAREST", "DEFAULT", 75, "OVERWRITE")
        cn_raster = None;precip_rx1day_raster = None;rainfall = None
        runoff_raster_mm = None;mean_intensity_raster = None;runoff_intensity_raster = None
        num += 1
        print(f"{num}. Рассчитана интенсивность стока с ячейки после ливня продолжительностью {дождь_расчетный_продолжительность} ч")

    # if task & (CalcTask.flow_dir | CalcTask.flow_acc):
    #     # Альтернативный способ расчет направления и накопления (с учетом веса стока)
    #     # !!! Не хватает памяти для запуска DeriveContinuousFlow !!!
    #     # Используем D8, так как StreamOrder требует именно этот тип
    #     out_accumulation_raster = arcpy.sa.DeriveContinuousFlow(
    #         in_surface_raster=dem,
    #         in_depressions_data=None,
    #         in_weight_raster=runoff,
    #         out_flow_direction_raster=flow_dir,
    #         flow_direction_type="D8",
    #         force_flow="NORMAL"
    #     )

    # Расчет направления стока
    if РасчетнаяЗадача.сток_направление in задача:
        out_flow_direction_raster = arcpy.sa.FlowDirection(
            in_surface_raster=цмр,
            force_flow="NORMAL",
            out_drop_raster=None,
            flow_direction_type="D8"
        )

    # Накопление стока
    if РасчетнаяЗадача.сток_накопление_rx5day in задача:
        out_accumulation_rx5day_raster = arcpy.sa.FlowAccumulation(
            in_flow_direction_raster=сток_направление,
            in_weight_raster=сток_rx5day_м,
            data_type="DOUBLE",
            flow_direction_type="D8"
        )
    
    # Сохранения направлений и суммы стоков выполнено в одном месте, чтобы не дублировать для разных вариантов
    if РасчетнаяЗадача.сток_направление in задача:
        out_flow_direction_raster.save(сток_направление)
        arcpy.management.BuildPyramids(сток_направление, -1, "NONE", "NEAREST", "DEFAULT", 75, "OVERWRITE")
        out_flow_direction_raster = None
        num += 1
        print(f"{num}. Рассчитано направление стока")
    if РасчетнаяЗадача.сток_накопление_rx5day in задача:
        out_accumulation_rx5day_raster.save(сток_накопление_rx5day)
        arcpy.management.BuildPyramids(сток_накопление_rx5day, -1, "NONE", "NEAREST", "DEFAULT", 75, "OVERWRITE")
        out_accumulation_rx5day_raster = None
        num += 1
        print(f"{num}. Рассчитан суммарный сток")

    # Расчет расхода воды при ливнях
    if РасчетнаяЗадача.ливень_расход in задача:
        сток_направление_raster = Raster(сток_направление)
        ширина_ячейки = сток_направление_raster.meanCellWidth
        сток_направление_raster = None
        
        # Сумма интенсивностей всех ячеек выше по течению
        rainfall_intensity_acc_raster = arcpy.sa.FlowAccumulation(
            in_flow_direction_raster=сток_направление,
            in_weight_raster=ливень_интенсивность,
            data_type="DOUBLE",
            flow_direction_type="D8"
        )
        # Удельный расход q (мм × м/с)
        удельный_расход_растр = rainfall_intensity_acc_raster * ширина_ячейки # Не перевожу в метры чтобы не потерять точность

        удельный_расход_растр.save(ливень_расход_удельный)
        arcpy.management.BuildPyramids(удельный_расход_растр, -1, "NONE", "NEAREST", "DEFAULT", 75, "OVERWRITE")
        rainfall_intensity_acc_raster = None;удельный_расход_растр = None
        num += 1
        print(f"{num}. Рассчитан удельный расход")


    # Выделение водотоков (Threshold)
    if РасчетнаяЗадача.водотоки_выявить_rx5day in задача:
        watercourse_rx5day_raster = Con(Raster(сток_накопление_rx5day) >= сток_руслоформирования, Raster(сток_накопление_rx5day))
        watercourse_rx5day_raster.save(водотоки_неупоряд_rx5day)
        arcpy.management.BuildPyramids(водотоки_неупоряд_rx5day, -1, "NONE", "NEAREST", "DEFAULT", 75, "OVERWRITE")
        watercourse_rx5day_raster = None
        num += 1
        print(f"{num}. Рассчитаны основные водотоки")

    # Порядок рек (Strahler)
    if РасчетнаяЗадача.водотоки_rx5day in задача:
        stream_rx5day_raster = arcpy.sa.StreamOrder(
            in_stream_raster=водотоки_неупоряд_rx5day,
            in_flow_direction_raster=сток_направление,
            order_method="STRAHLER"
        )
        stream_rx5day_raster.save(водотоки_rx5day)
        arcpy.management.BuildPyramids(водотоки_rx5day, -1, "NONE", "NEAREST", "DEFAULT", 75, "OVERWRITE")
        stream_rx5day_raster = None
        num += 1
        print(f"{num}. Рассчитан порядок притоков водотоков (Не обязательно)")

    # Векторизация рек
    if РасчетнаяЗадача.водотоки_векторизовать_rx5day in задача:
        arcpy.sa.StreamToFeature(
            in_stream_raster=водотоки_rx5day,
            in_flow_direction_raster=сток_направление,
            out_polyline_features=водотоки_rx5day_v,
            simplify="SIMPLIFY"
        )
        num += 1
        print(f"{num}. Водотоки векторизованы")

    # Комбинировать водотоки и океан
    if РасчетнаяЗадача.водотоки_и_океан in задача:
        ocean_raster_temp = r"memory\ocean_raster_temp"

        # Используем PolygonToRaster. Важно: cell_size берем из ЦМР
        arcpy.conversion.PolygonToRaster(
            in_features=океан,
            value_field="OBJECTID",      # Любое поле, главное наличие
            out_rasterdataset=ocean_raster_temp,
            cell_assignment="CELL_CENTER",
            cellsize=цмр
        )

        # Превращаем в бинарный растр (1 - вода, NoData - суша)
        ocean_binary = Con(~IsNull(Raster(ocean_raster_temp)), 1)

        # Объединение: (Реки) ИЛИ (Океан)
        # Логика: Если пиксель есть в Реках ИЛИ в Океане -> ставим 1.
        combined_sources = Con(
            (~IsNull(Raster(водотоки_rx5day))) | (~IsNull(ocean_binary)),
            1
        )

        combined_sources.save(водотоки_и_океан)
        arcpy.management.BuildPyramids(водотоки_и_океан, -1, "NONE", "NEAREST", "DEFAULT", 75, "OVERWRITE")

        # Чистка памяти
        ocean_binary = None; combined_sources = None
        arcpy.management.Delete(ocean_raster_temp)
                
        num += 1
        print(f"{num}. Речные водотоки объединены с океаном")

    # Stream Burn-in (понижаем русла на 50 метров). Думал, что может быть пригодиться, но не пригодилось.
    if РасчетнаяЗадача.цмр_прожиг in задача:
        dem_burn_raster = Con(IsNull(Raster(водотоки_и_океан)), Raster(цмр), Raster(цмр) - 50)    
        dem_burn_raster.save(цмр_burn)
        arcpy.management.BuildPyramids(цмр_burn, -1, "NONE", "NEAREST", "DEFAULT", 75, "OVERWRITE")
        dem_burn_raster = None
        num += 1
        print(f"{num}. Прожик водотоков на УМР")

    # Построение HAND
    if РасчетнаяЗадача.hand in задача:
        tool = arcpy.ImportToolbox(расчет_hand_инструмент_путь,"NewToolbox")
        tool.Model21(
            Input_DEM=цмр,
            Input_Stream_Raster=водотоки_rx5day,
            Temp_Folder=temp,
            Output_HAND_Raster=hand
        )
        arcpy.management.BuildPyramids(hand, -1, "NONE", "NEAREST", "DEFAULT", 75, "OVERWRITE")
        num += 1
        print(f"{num}. HAND рассчитан")

    # Растр шероховатости поверхности по WorldCover
    if РасчетнаяЗадача.шероховатость in задача:
        # ПЕРЕКЛАССИФИКАЦИЯ В ЦЕЛЫЕ ЧИСЛА, т.к Reclassify поддерживает целые числа
        remap_int_list = [(c, int(v * 1000)) for (c,v) in покрытие_шероховатость_перекодировка]
        remap_obj = RemapValue(remap_int_list)
        manning_int = Reclassify(Raster(покрытие), "Value", remap_obj, missing_values="NODATA")
    
        # КОНВЕРТАЦИЯ ОБРАТНО ВО FLOAT    
        # Используем алгебру карт для деления
        manning_raster = manning_int / 1000.0
        manning_raster.save(маннинг)
        arcpy.management.BuildPyramids(маннинг, -1, "NONE", "NEAREST", "DEFAULT", 75, "OVERWRITE")
        manning_raster = None
        num += 1
        print(f"{num}. Растр шершавости покрытий рассчитан")

    # Расчет гидравлического радиуса (глубины h для склонового стока)
    if РасчетнаяЗадача.ливень_глубина_стока in задача:
        n_rast = Raster(маннинг)
        q_rast = Raster(ливень_расход_удельный) / 1_000.

        # Добавляем маленькое число (0.0001), чтобы избежать деления на ноль
        s_rast = Raster(уклон) / 100. + 0.00001

        # Формула глубины h = ( (n * q) / sqrt(s) )^0.6
        h_rast = Power((n_rast * q_rast) / SquareRoot(s_rast), 0.6)
        # Корректировка: ставим минимальную глубину (например, 0.01 м), чтобы избежать возможных ошибок
        h_rast = Con(h_rast < 0.01, 0.01, h_rast)
        
        h_rast.save(глубина)
        arcpy.management.BuildPyramids(h_rast, -1, "NONE", "NEAREST", "DEFAULT", 75, "OVERWRITE")
        n_rast = None;q_rast = None; s_rast = None; h_rast = None
        num += 1
        print(f"{num}. Глубина склонового стока рассчитана")

    # Поверхность стоимости для расчета расстояний до водотоков
    if РасчетнаяЗадача.поверхность_стоимости in задача:
        n_rast = Raster(маннинг)

        # Формула: (v^2 * n^2) / h^(4/3)
        cost_raster = (поток_скорость_цель ** 2) * (n_rast ** 2) / (поток_глубина_цель ** (4./3.))
        cost_raster.save(стоимость_горизонтальная)
        arcpy.management.BuildPyramids(стоимость_горизонтальная, -1, "NONE", "NEAREST", "DEFAULT", 75, "OVERWRITE")
        n_rast = None;cost_raster = None
        num += 1
        print(f"{num}. Поверхность стоимости для расчета расстояний до водотоков готова")
    
    # Вертикальный фактор
    if РасчетнаяЗадача.вертикальный_фактор in задача:
        create_vf_table(гидро_уклон_база, Path(таблица_VF))
        num += 1
        print(f"{num}. Таблица вертикального фактора готова")
    
    # Бассейны
    if РасчетнаяЗадача.бассейны in задача:
        bassin_raster = arcpy.sa.Basin(
            in_flow_direction_raster=сток_направление
        )
        bassin_raster.save(bassin)
        arcpy.management.BuildPyramids(bassin, -1, "NONE", "NEAREST", "DEFAULT", 75, "OVERWRITE")
        bassin_raster = None
        num += 1
        print(f"{num}. Бассейны рассчитаны")

    # Векторизовать бассейны
    if РасчетнаяЗадача.бассейны_веторизовать in задача:
        arcpy.conversion.RasterToPolygon(
            in_raster=bassin,
            out_polygon_features=bassin_v,
            simplify="SIMPLIFY",
            raster_field="Value",
            create_multipart_features="SINGLE_OUTER_PART",
            max_vertices_per_feature=None
        )
        num += 1
        print(f"{num}. Бассейны векторизованы")


    print(f"1 этап расчетов выполнен")
    

1. Рассчитана интенсивность стока с ячейки после ливня продолжительностью 1.0 ч
2. Рассчитан удельный расход
3. Глубина склонового стока рассчитана
1 этап расчетов выполнен


Дальше нужно вручную [генерализовать](https://pro.arcgis.com/ru/pro-app/3.5/tool-reference/data-management/eliminate.htm) водосборы меньше определенной площади, а также вручную разбить водотоками водосборы площадью свыше ≈100 тыс. км² (при разрешении ≈30 м).

In [8]:
with ((arcpy.EnvManager(outputCoordinateSystem=arcpy.SpatialReference(32631), cellSize=цмр, scratchWorkspace=temp))):
    if РасчетнаяЗадача.накопление_расстояния in задача:
        num2 = 0

        Path(стоимость_гор_тайлы).mkdir(parents=True, exist_ok=True)
        Path(стоимость_накопл_тайлы).mkdir(parents=True, exist_ok=True)
        
        # Разбиваем растр стоимости по границам бассейнов, т.к. расчет на большую территорию компьютер не тянет
        arcpy.management.SplitRaster(
            in_raster=стоимость_горизонтальная,
            out_folder=стоимость_гор_тайлы,
            out_base_name="cost_",
            split_method="POLYGON_FEATURES",
            format="TIFF",
            resampling_type="NEAREST",
            overlap=3,  # Перекрытие в 3 пикселя для склейки
            units="PIXELS",
            split_polygon_feature_class=бассейны_генер,
        )

        vf_object = VfTable(таблица_VF)

        # Определяем список растровых расширений
        raster_extensions = ('.tif', '.tiff', '.img', '.grd')
        
        # Получаем список файлов
        split_files = [f for f in Path(стоимость_гор_тайлы).glob('*') if f.suffix.lower() in raster_extensions]
        # Сортируем по размеру файла чтобы как можно раньше предварительно посмотреть результаты
        split_files.sort(key=lambda f: Path(f).stat().st_size)
        print(f"Найдено тайлов для обработки: {len(split_files)}")
                
        # Выполняем расчет по бассейнам
        processed_files = [] # Сюда будем складывать пути к готовым кускам
        for raster in split_files:
            # Формируем имена выходных файлов
            out_acc_name = f"{raster.stem}_acc.tif"
            out_dir_name = f"{raster.stem}_dir.tif"

            out_acc_path = Path(стоимость_накопл_тайлы) / out_acc_name
            out_dir_path = Path(стоимость_накопл_тайлы) / out_dir_name

            print(f"\t{num}.{num2}. Обработка тайла: {raster.name}")
            
            out_distance_accumulation_raster = arcpy.sa.DistanceAccumulation(
                in_source_data=водотоки_rx5day,
                in_surface_raster=цмр,
                in_cost_raster=str(raster),
                in_vertical_raster=цмр,
                vertical_factor=vf_object,
                out_back_direction_raster=str(out_dir_path),
                distance_method="PLANAR"
            )
            out_distance_accumulation_raster.save(str(out_acc_path))
            arcpy.management.BuildPyramids(str(out_acc_path), -1, "NONE", "CUBIC", "DEFAULT", 75, "OVERWRITE")
            processed_files.append(str(out_acc_path))
            
            num2 += 1
            print(f"\t\t. Готово")

        if processed_files:
            print("Сборка мозаики...")
            # Определяем тип пикселя по первому файлу
            first_raster = Raster(processed_files[0])
            pixel_type_str = get_arcpy_pixel_type(first_raster)
        
            # Объединяем в единую мозаику
            arcpy.management.MosaicToNewRaster(
                input_rasters=";".join(processed_files),
                output_location=str(Path(накопленный_напор).parent),
                raster_dataset_name_with_extension=str(Path(накопленный_напор).name),
                pixel_type=pixel_type_str,
                cellsize=None,
                number_of_bands=1,
                mosaic_method="MINIMUM", # Поскольку мы считаем накопленную стоимость (энергию), правильнее брать минимальное значение из перекрывающихся тайлов
                mosaic_colormap_mode="FIRST"
            )
            
            num += 1
            print(f"{num}. Мозаика завершена: {накопленный_напор}")
        else:
            print("Нечего объединять, список обработанных файлов пуст.")

print(f"Гидрологический анализ завершен")

Гидрологический анализ завершен
